# Popularity Baseline for MIND Ranking


Prerequisites:
- The MIND data is available under `data/train` and `data/valid`.
- `pandas`, `numpy`, and `scikit-learn` are available in the environment.

Learning goals:
- Build a global news popularity score from training clicks.
- Rank validation candidates by this popularity score.
- Evaluate ranking with `AUC`, `MRR`, `nDCG@5`, and `nDCG@10`.
- Produce a simple sanity-check baseline for the full pipeline.


In [1]:
from __future__ import annotations

import math
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "data").exists():
            return candidate
    raise FileNotFoundError(
        f"Could not find project root from {start}. Expected a parent directory containing 'data'."
    )

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = find_project_root(NOTEBOOK_DIR)

TRAIN_BEHAVIORS_PATH = PROJECT_ROOT / "data/train/behaviors.tsv"
VALID_BEHAVIORS_PATH = PROJECT_ROOT / "data/valid/behaviors.tsv"
OUTPUT_DIR = NOTEBOOK_DIR / "popularity_baseline_output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for path in [TRAIN_BEHAVIORS_PATH, VALID_BEHAVIORS_PATH]:
    if not path.exists():
        raise FileNotFoundError(f"Missing required file: {path}")

print("Notebook directory:", NOTEBOOK_DIR.resolve())
print("Project root:", PROJECT_ROOT.resolve())
print("Train behaviors:", TRAIN_BEHAVIORS_PATH)
print("Valid behaviors:", VALID_BEHAVIORS_PATH)
print("Output directory:", OUTPUT_DIR.resolve())


Notebook directory: /Users/xiangningdeng/Desktop/2026 UCLA MDSH-Submit/round1_baselines
Project root: /Users/xiangningdeng/Desktop/2026 UCLA MDSH-Submit
Train behaviors: /Users/xiangningdeng/Desktop/2026 UCLA MDSH-Submit/data/train/behaviors.tsv
Valid behaviors: /Users/xiangningdeng/Desktop/2026 UCLA MDSH-Submit/data/valid/behaviors.tsv
Output directory: /Users/xiangningdeng/Desktop/2026 UCLA MDSH-Submit/round1_baselines/popularity_baseline_output


In [2]:
BEHAVIORS_COLUMNS = ["impression_id", "user_id", "time", "history", "impressions"]

train_df = pd.read_csv(TRAIN_BEHAVIORS_PATH, sep="\t", header=None, names=BEHAVIORS_COLUMNS, dtype=str)
valid_df = pd.read_csv(VALID_BEHAVIORS_PATH, sep="\t", header=None, names=BEHAVIORS_COLUMNS, dtype=str)

print("Train shape:", train_df.shape)
print("Valid shape:", valid_df.shape)
train_df.head(2)


Train shape: (156965, 5)
Valid shape: (73152, 5)


,impression_id,user_id,time,history,impressions
0,1,U13740,11/11/2019 9:05:58 AM,N55189 N42782 N34694 N45794 N18445 N63302 N104...,N55689-1 N35729-0
1,2,U91836,11/12/2019 6:11:30 PM,N31739 N6072 N63045 N23979 N35656 N43353 N8129...,N20678-0 N39317-0 N58114-0 N20495-0 N42977-0 N...


In [3]:
def parse_impressions(impressions_str: str):
    news_ids = []
    labels = []
    if not isinstance(impressions_str, str) or not impressions_str.strip():
        return news_ids, labels
    for item in impressions_str.split():
        if "-" not in item:
            continue
        news_id, label = item.rsplit("-", 1)
        try:
            labels.append(int(label))
            news_ids.append(news_id)
        except ValueError:
            continue
    return news_ids, labels

news_click_count = Counter()
for impressions in train_df["impressions"]:
    ids, labels = parse_impressions(impressions)
    for nid, label in zip(ids, labels):
        if label == 1:
            news_click_count[nid] += 1

print("Unique clicked news in train:", len(news_click_count))
print("Top 10 popular news:", news_click_count.most_common(10))


Unique clicked news in train: 7713
Top 10 popular news: [('N55689', 4316), ('N35729', 3346), ('N33619', 3246), ('N53585', 2835), ('N63970', 2578), ('N49685', 2294), ('N49279', 2270), ('N287', 2128), ('N23446', 1930), ('N51048', 1875)]


In [4]:
def score_by_popularity(news_ids, click_counter):
    return [click_counter.get(news_id, 0) for news_id in news_ids]

all_labels = []
all_scores = []

for impressions in valid_df["impressions"]:
    news_ids, labels = parse_impressions(impressions)
    scores = score_by_popularity(news_ids, news_click_count)
    if labels:
        all_labels.append(labels)
        all_scores.append(scores)

def mean_auc(group_labels, group_scores):
    aucs = []
    for labels, scores in zip(group_labels, group_scores):
        if len(set(labels)) < 2:
            continue
        aucs.append(roc_auc_score(labels, scores))
    return float(np.mean(aucs)) if aucs else 0.0

def mrr_score(labels, scores):
    order = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)
    for rank, idx in enumerate(order, start=1):
        if labels[idx] == 1:
            return 1.0 / rank
    return 0.0

def mean_mrr(group_labels, group_scores):
    values = [mrr_score(labels, scores) for labels, scores in zip(group_labels, group_scores)]
    return float(np.mean(values)) if values else 0.0

def dcg_at_k(labels_sorted, k):
    dcg = 0.0
    for i in range(min(k, len(labels_sorted))):
        rel = labels_sorted[i]
        dcg += (2 ** rel - 1) / math.log2(i + 2)
    return dcg

def ndcg_at_k(labels, scores, k):
    order = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)
    ranked_labels = [labels[i] for i in order]
    dcg = dcg_at_k(ranked_labels, k)
    idcg = dcg_at_k(sorted(labels, reverse=True), k)
    return 0.0 if idcg == 0 else dcg / idcg

def mean_ndcg(group_labels, group_scores, k):
    values = [ndcg_at_k(labels, scores, k) for labels, scores in zip(group_labels, group_scores)]
    return float(np.mean(values)) if values else 0.0

metrics = {
    "AUC": mean_auc(all_labels, all_scores),
    "MRR": mean_mrr(all_labels, all_scores),
    "nDCG@5": mean_ndcg(all_labels, all_scores, 5),
    "nDCG@10": mean_ndcg(all_labels, all_scores, 10),
}
metrics


{'AUC': 0.531751528369352,
 'MRR': 0.2671452238107854,
 'nDCG@5': 0.24604210472429233,
 'nDCG@10': 0.3098167064622724}

In [6]:
metrics_df = pd.DataFrame([metrics])
metrics_path = OUTPUT_DIR / "metrics_popularity_baseline.csv"
metrics_df.to_csv(metrics_path, index=False)

print("=== Validation Results ===")
for key, value in metrics.items():
    print(f"{key}: {value:.4f}")
metrics_df


=== Validation Results ===
AUC: 0.5318
MRR: 0.2671
nDCG@5: 0.2460
nDCG@10: 0.3098


,AUC,MRR,nDCG@5,nDCG@10
0,0.531752,0.267145,0.246042,0.309817
